# 🦷 Dental X-Ray Detection (YOLO) — Kaggle auto-download version

Detects **Fillings, Implant, Cavity, Impacted Tooth** on panoramic dental X-rays and adds a rule-based **treatment suggestion** layer.

**What changed vs. the old notebook**
- Dataset comes straight from Kaggle via `kagglehub` (no manual zip upload).
- Bigger model (YOLO11m) + larger image size, cosine LR, early stopping.
- Rare classes (Cavity, Impacted Tooth) are oversampled **in train only**.
- Boxes are validated/clipped before conversion to YOLO format.
- Final evaluation on the **test** split, plus automatic best confidence threshold.

> Runtime → Change runtime type → **GPU (T4)** before running.
> Research/education tool only. Not a clinical diagnostic device.

## 1. Setup

In [ ]:
!pip install -q ultralytics kagglehub

import os, glob, shutil, random, yaml, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Download dataset directly from Kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("imtkaggleteam/dental-radiography")

print("Path to dataset files:", path)

In [ ]:
# Locate train / valid / test folders (each contains images + _annotations.csv)
def find_split_dirs(root):
    splits = {}
    for dirpath, dirnames, filenames in os.walk(root):
        if "_annotations.csv" in filenames:
            name = os.path.basename(dirpath).lower()
            if name in ("train", "valid", "val", "test"):
                splits["valid" if name == "val" else name] = dirpath
    return splits

SPLITS = find_split_dirs(path)
for k, v in SPLITS.items():
    print(f"{k:6s} -> {v}")

assert {"train", "valid", "test"} <= set(SPLITS), \
    "Could not find train/valid/test folders. Check the printed dataset path structure."

## 3. Explore the data

In [ ]:
train_csv = pd.read_csv(os.path.join(SPLITS["train"], "_annotations.csv"))
train_csv.columns = [c.strip() for c in train_csv.columns]
display(train_csv.head())
train_csv.info()

In [ ]:
print(train_csv["class"].value_counts())
for s in ["train", "valid", "test"]:
    n_img = len([f for f in os.listdir(SPLITS[s]) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    print(f"{s:6s}: {n_img} images")

In [ ]:
# Ground-truth boxes on one sample image
sample_name = "0674_jpg.rf.e02a155a0c135687b9301ff9a20d220a.jpg"
if not os.path.exists(os.path.join(SPLITS["train"], sample_name)):
    sample_name = train_csv.filename.iloc[0]

img = cv2.cvtColor(cv2.imread(os.path.join(SPLITS["train"], sample_name)), cv2.COLOR_BGR2RGB)
for _, r in train_csv[train_csv.filename == sample_name].iterrows():
    cv2.rectangle(img, (int(r.xmin), int(r.ymin)), (int(r.xmax), int(r.ymax)), (0, 255, 0), 1)
    cv2.putText(img, r["class"], (int(r.xmin), max(10, int(r.ymin) - 3)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

plt.figure(figsize=(10, 6)); plt.imshow(img); plt.axis("off"); plt.title(sample_name[:30])
plt.show()

## 4. Convert CSV → YOLO format (with validation + rare-class oversampling)

In [ ]:
CLASS_NAMES = ["Fillings", "Implant", "Cavity", "Impacted Tooth"]
CLS2ID = {n: i for i, n in enumerate(CLASS_NAMES)}

OVERSAMPLE_RARE = True   # train split only; Cavity x3, Impacted Tooth x2
BASE = "/content/dental_dataset"

if os.path.exists(BASE):
    shutil.rmtree(BASE)
for s in ["train", "valid", "test"]:
    os.makedirs(f"{BASE}/images/{s}", exist_ok=True)
    os.makedirs(f"{BASE}/labels/{s}", exist_ok=True)

IMG_EXT = (".jpg", ".jpeg", ".png")

def build_split(split):
    src = SPLITS[split]
    df = pd.read_csv(os.path.join(src, "_annotations.csv"))
    df.columns = [c.strip() for c in df.columns]

    unknown = set(df["class"].unique()) - set(CLASS_NAMES)
    assert not unknown, f"Unknown classes in {split}: {unknown}"

    n_img = n_box = n_skip = n_dup = 0
    for fname, g in tqdm(df.groupby("filename"), desc=split):
        src_img = os.path.join(src, fname)
        if not os.path.exists(src_img):
            continue

        lines, present = [], set()
        for _, r in g.iterrows():
            w, h = float(r.width), float(r.height)
            x1 = max(0.0, min(w, float(r.xmin))); x2 = max(0.0, min(w, float(r.xmax)))
            y1 = max(0.0, min(h, float(r.ymin))); y2 = max(0.0, min(h, float(r.ymax)))
            if (x2 - x1) < 2 or (y2 - y1) < 2:
                n_skip += 1
                continue
            cid = CLS2ID[r["class"]]
            present.add(cid)
            lines.append(f"{cid} {((x1+x2)/2)/w:.6f} {((y1+y2)/2)/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
        if not lines:
            continue

        copies = 1
        if OVERSAMPLE_RARE and split == "train":
            if 2 in present:   copies += 2
            elif 3 in present: copies += 1

        stem, ext = os.path.splitext(fname)
        for k in range(copies):
            out = stem if k == 0 else f"{stem}_dup{k}"
            shutil.copy(src_img, f"{BASE}/images/{split}/{out}{ext}")
            with open(f"{BASE}/labels/{split}/{out}.txt", "w") as f:
                f.write("\n".join(lines) + "\n")
        n_img += 1; n_box += len(lines); n_dup += copies - 1

    # images without any annotation rows -> background images (no label file)
    labelled = set(df["filename"])
    for fname in os.listdir(src):
        if fname.lower().endswith(IMG_EXT) and fname not in labelled:
            shutil.copy(os.path.join(src, fname), f"{BASE}/images/{split}/{fname}")

    print(f"{split}: {n_img} images, {n_box} boxes, {n_skip} degenerate boxes skipped, {n_dup} oversampled copies")

for s in ["train", "valid", "test"]:
    build_split(s)

for s in ["train", "valid", "test"]:
    print(s, "| images:", len(os.listdir(f"{BASE}/images/{s}")), "| labels:", len(os.listdir(f"{BASE}/labels/{s}")))

In [ ]:
# Verify: draw YOLO labels back on an image (sanity check of the conversion)
lbl_path = f"{BASE}/labels/train/{os.path.splitext(sample_name)[0]}.txt"
img = cv2.cvtColor(cv2.imread(f"{BASE}/images/train/{sample_name}"), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
for line in open(lbl_path):
    c, x, y, bw, bh = line.split(); c = int(c); x, y, bw, bh = map(float, (x, y, bw, bh))
    p1 = (int((x - bw/2) * w), int((y - bh/2) * h)); p2 = (int((x + bw/2) * w), int((y + bh/2) * h))
    cv2.rectangle(img, p1, p2, (0, 255, 0), 1)
    cv2.putText(img, CLASS_NAMES[c], (p1[0], max(10, p1[1] - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)
plt.figure(figsize=(10, 6)); plt.imshow(img); plt.axis("off"); plt.show()

In [ ]:
data_cfg = {
    "path": BASE,
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}
with open("/content/dataset.yaml", "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print(open("/content/dataset.yaml").read())

## 5. Train
Tuning knobs are in the first lines. If you hit **CUDA out of memory**, set `BATCH = 8` or `IMGSZ = 640`.
For a faster run, use `MODEL = "yolo11s.pt"`.

In [ ]:
from ultralytics import YOLO

MODEL  = "yolo11m.pt"   # yolo11s.pt = faster, yolo11l.pt = bigger
IMGSZ  = 800            # originals are 512x256; upscaling helps small cavities
EPOCHS = 50
BATCH  = 16

model = YOLO(MODEL)

results = model.train(
    data="/content/dataset.yaml",
    epochs=EPOCHS,
    patience=20,
    imgsz=IMGSZ,
    batch=BATCH,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=0.0005,
    # augmentation tuned for grayscale panoramic X-rays
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.3,
    degrees=5, translate=0.1, scale=0.4,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, close_mosaic=10, mixup=0.0,
    cache="ram",
    workers=2,
    seed=SEED,
    project="Dental_Project",
    name="Dental_Xray_Model",
    exist_ok=True,
    plots=True,
)

## 6. Training curves & confusion matrix

In [ ]:
RUN_DIR = str(results.save_dir)
print("Run dir:", RUN_DIR)
display(Image.open(f"{RUN_DIR}/results.png"))

In [ ]:
display(Image.open(f"{RUN_DIR}/confusion_matrix_normalized.png"))

## 7. Evaluate best weights on validation **and test** sets

In [ ]:
best_path = f"{RUN_DIR}/weights/best.pt"
best = YOLO(best_path)
print(best.names)

print("=== VALIDATION ===")
mv = best.val(data="/content/dataset.yaml", split="val", imgsz=IMGSZ, plots=False)
print(f"mAP50={mv.box.map50:.3f}  mAP50-95={mv.box.map:.3f}  P={mv.box.mp:.3f}  R={mv.box.mr:.3f}")

print("=== TEST (unseen) ===")
mt = best.val(data="/content/dataset.yaml", split="test", imgsz=IMGSZ, plots=False)
print(f"mAP50={mt.box.map50:.3f}  mAP50-95={mt.box.map:.3f}  P={mt.box.mp:.3f}  R={mt.box.mr:.3f}")

In [ ]:
# Best confidence threshold from the F1 curve (validation set)
try:
    f1 = mv.box.f1_curve.mean(0)
    BEST_CONF = float(np.clip(mv.box.px[f1.argmax()], 0.15, 0.6))
except Exception as e:
    print("Could not read F1 curve:", e)
    BEST_CONF = 0.25
print("Using confidence threshold:", round(BEST_CONF, 3))

## 8. Predict on an image

In [ ]:
test_imgs = sorted(glob.glob(f"{BASE}/images/test/*.jpg"))
TEST_IMAGE = test_imgs[0]          # or set your own path
print(TEST_IMAGE)

res = best.predict(TEST_IMAGE, conf=BEST_CONF, imgsz=IMGSZ, augment=True, verbose=False)[0]

plt.figure(figsize=(12, 7))
plt.imshow(res.plot()[..., ::-1]); plt.axis("off"); plt.show()

for b in res.boxes:
    print(f"{best.names[int(b.cls[0])]:15s} : {float(b.conf[0]):.2f}")

## 9. Findings → suggested treatment (rule-based layer)

In [ ]:
# The detector only finds & labels what is visible. Suggestions below are
# simple rules for decision support -- a dentist must review every result.
SUGGESTIONS = {
    "Cavity":         "Possible caries: clinical exam + restoration (filling) evaluation.",
    "Fillings":       "Existing restoration: monitor for marginal leakage / secondary decay.",
    "Implant":        "Existing implant: check bone level and peri-implant health at recall.",
    "Impacted Tooth": "Impacted tooth: refer for surgical / orthodontic assessment.",
}

def analyse_image(img_path, conf=None):
    conf = BEST_CONF if conf is None else conf
    r = best.predict(img_path, conf=conf, imgsz=IMGSZ, augment=True, verbose=False)[0]
    found = {}
    for b in r.boxes:
        name = best.names[int(b.cls[0])]
        found.setdefault(name, []).append(float(b.conf[0]))
    print(f"Image: {os.path.basename(img_path)}")
    if not found:
        print("  No findings above threshold."); return r
    for name, confs in found.items():
        print(f"  - {name} x{len(confs)} (max conf {max(confs):.2f}): {SUGGESTIONS[name]}")
    print("  ! Decision support only. Confirm with a qualified dentist.")
    return r

_ = analyse_image(TEST_IMAGE)

## 10. Download the trained model

In [ ]:
from google.colab import files
files.download(best_path)

# Optional: also keep a copy in Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy(best_path, '/content/drive/MyDrive/dental_best.pt')

### Notes
- Your first run reached overall mAP50 ≈ 0.77 on validation (Cavity ≈ 0.39). This notebook is set up to improve mainly Cavity, but I can't guarantee a specific number — check the **test** metrics in section 7.
- Cavity is the hardest class in every public dental dataset (few, small, low-contrast examples).
- Training (50 epochs) takes roughly 30–50 min on a free T4 with the defaults.